# 01 - Feast Feature Store (Remote Client)

## Data Scientist Workflow

This notebook uses the **Feast Operator's remote client config** automatically mounted at:
```
/opt/app-root/src/feast-config/salesforecasting
```

### Architecture

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     FEAST OPERATOR MANAGED SERVICES                         │
├─────────────────────────────────────────────────────────────────────────────┤
│  ┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐      │
│  │  Registry Service│    │  Offline Service │    │  Online Service  │      │
│  │  (gRPC :443)     │    │  (gRPC :443)     │    │  (gRPC :443)     │      │
│  └────────┬─────────┘    └────────┬─────────┘    └────────┬─────────┘      │
│           │                       │                       │                │
│           ▼                       ▼                       ▼                │
│      PostgreSQL              Ray Cluster               Redis               │
└─────────────────────────────────────────────────────────────────────────────┘
                                    │
┌───────────────────────────────────┼─────────────────────────────────────────┐
│                     DATA SCIENTIST NOTEBOOK                                 │
│           ┌───────────────────────▼───────────────────────┐                │
│           │  Feast SDK (remote client)                    │                │
│           │  • get_online_features() → Online Service     │                │
│           │  • get_historical_features() → Offline Service│                │
│           │  • list_*() → Registry Service                │                │
│           └───────────────────────────────────────────────┘                │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Prerequisites (Admin)

```bash
oc apply -k manifests/
```

---
## Step 0: Install Dependencies

In [ ]:
%pip install -q "feast==0.61.0" pandas pyarrow

---
## Step 1: Configure Remote Connection

Connect to admin-deployed services using service account authentication.

In [ ]:
import os
from pathlib import Path

# Feast Operator provides client config at this path
FEAST_CONFIG_PATH = "/opt/app-root/src/feast-config/salesforecasting"

# Local paths for data
SHARED_ROOT = Path("/opt/app-root/src/shared")
DATA_DIR = SHARED_ROOT / "data"

# Verify config exists
if Path(FEAST_CONFIG_PATH).exists():
    print(f"✅ Feast config found: {FEAST_CONFIG_PATH}")
    with open(FEAST_CONFIG_PATH) as f:
        print(f"\n📄 Config content:\n{f.read()}")
else:
    print(f"❌ Feast config not found at {FEAST_CONFIG_PATH}")
    print("   Make sure OpenShift AI Feature Store connection is configured")

In [ ]:
# In remote client mode, the SDK communicates with Feast server services
# No direct Ray/Redis/Postgres connection needed from notebook
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Data directory: {DATA_DIR}")

---
## Step 2: Connect to Feast (Remote Client)

Initialize the Feast SDK using the operator-provided config.

In [ ]:
from feast import FeatureStore

# Initialize Feast SDK with operator-provided config
store = FeatureStore(repo_path=FEAST_CONFIG_PATH)

print(f"✅ Connected to Feast Feature Store")
print(f"   Project: {store.project}")
print(f"   Config: {FEAST_CONFIG_PATH}")

In [ ]:
# List registered objects from remote registry
print("📋 Registered Objects (via Registry Service):")
print("-" * 50)

entities = store.list_entities()
feature_views = store.list_feature_views()
feature_services = store.list_feature_services()

print(f"   Entities: {[e.name for e in entities]}")
print(f"   Feature Views: {[fv.name for fv in feature_views]}")
print(f"   Feature Services: {[fs.name for fs in feature_services]}")

if not entities:
    print("\n⚠️ No features registered yet. Admin needs to run 'feast apply'.")

---
## Step 3: Generate Sample Data

Create synthetic sales data for feature retrieval examples.

In [ ]:
# Note: In remote client mode, we use the operator-provided config
# No need to generate feature_store.yaml - it's managed by the Feast Operator
# 
# Features are registered by admin using 'feast apply' from the server pod
# This notebook focuses on CONSUMING features via the remote SDK

print("ℹ️ Remote Client Mode:")
print("   - Config: Operator-provided (no local feature_store.yaml needed)")
print("   - Registry: Accessed via gRPC (feast-*-registry service)")
print("   - Online Store: Accessed via gRPC (feast-*-online service)")
print("   - Offline Store: Accessed via gRPC (feast-*-offline service)")

In [ ]:
# Create entity DataFrame for feature retrieval examples
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone

# Sample entity data for testing feature retrieval
entity_df = pd.DataFrame([
    {"store_id": 1, "dept_id": 1, "event_timestamp": datetime(2023, 6, 1, tzinfo=timezone.utc)},
    {"store_id": 10, "dept_id": 5, "event_timestamp": datetime(2023, 6, 15, tzinfo=timezone.utc)},
    {"store_id": 25, "dept_id": 10, "event_timestamp": datetime(2023, 7, 1, tzinfo=timezone.utc)},
    {"store_id": 45, "dept_id": 14, "event_timestamp": datetime(2023, 7, 15, tzinfo=timezone.utc)},
])

print("✅ Created entity DataFrame for feature retrieval:")
entity_df

---
## Step 4: Retrieve Online Features

Get feature values from the online store (Redis) via the Feast Online Service.

In [ ]:
# Get online features for real-time inference
# These come from Redis via the Feast Online Service

online_features = store.get_online_features(
    features=[
        "sales_features:weekly_sales",
        "sales_features:lag_1",
        "sales_features:rolling_mean_4w",
        "store_features:store_type",
        "store_features:store_size",
    ],
    entity_rows=[
        {"store_id": 1, "dept_id": 1},
        {"store_id": 10, "dept_id": 5},
    ]
).to_dict()

print("📊 Online Features (from Redis via Feast Online Service):")
print("-" * 60)
for k, v in online_features.items():
    print(f"   {k}: {v}")

---
## Step 5: Retrieve Historical Features

Get historical feature values from the offline store (Ray) via the Feast Offline Service.
This is used for training ML models.

In [ ]:
# Get historical features for training
# The entity_df specifies which entities and timestamps to retrieve features for
import time

print("📊 Historical Feature Retrieval (via Feast Offline Service):")
print("-" * 60)

t0 = time.time()
historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "sales_features:weekly_sales",
        "sales_features:lag_1",
        "sales_features:lag_2",
        "sales_features:rolling_mean_4w",
        "store_features:store_type",
        "store_features:store_size",
    ]
).to_df()
elapsed = time.time() - t0

print(f"✅ Retrieved {len(historical_features)} rows in {elapsed:.2f}s")
historical_features

In [ ]:
# Use features for ML training example
if len(historical_features) > 0:
    print("📊 Features ready for ML training:")
    print(f"   Shape: {historical_features.shape}")
    print(f"   Columns: {list(historical_features.columns)}")
    print(f"\n   Sample data:")
    display(historical_features.head())
else:
    print("⚠️ No historical features retrieved - check if data is materialized")

In [ ]:
# Save training data locally for ML workflow
if len(historical_features) > 0:
    historical_features.to_parquet(DATA_DIR / "training_features.parquet", index=False)
    print(f"✅ Saved training data: {DATA_DIR / 'training_features.parquet'}")
    print(f"   Ready for 02-training.ipynb")

---
## Note: Admin Responsibilities

The following operations require direct access (from Feast server pod, not remote client):
- `feast apply` - Register feature definitions
- `feast materialize` - Push features to online store

In [ ]:
# Admin commands (run from Feast server pod or with direct-connection config):
print("""
Admin Setup Commands:
─────────────────────
# From Feast server pod:
oc exec -it deploy/feast-salesforecasting-server -n feast-trainer-demo -- bash
cd /feature_repo
feast apply
feast materialize 2022-01-01T00:00:00 $(date -u +%Y-%m-%dT%H:%M:%S)

# Or use the direct-connection notebook (01-feast-features.ipynb)
# which generates its own feature_store.yaml with direct DB access
""")

---
## ✅ Complete!

### Remote Client Mode Summary

In [ ]:
print("""
┌─────────────────────────────────────────────────────────────┐
│              Remote Client Mode - What Works                │
├─────────────────────────────────────────────────────────────┤
│  ✅ store.list_entities()          → Registry Service       │
│  ✅ store.list_feature_views()     → Registry Service       │
│  ✅ store.get_online_features()    → Online Service         │
│  ✅ store.get_historical_features()→ Offline Service        │
├─────────────────────────────────────────────────────────────┤
│              Requires Direct Access (Admin)                 │
├─────────────────────────────────────────────────────────────┤
│  ❌ feast apply                    → Needs server pod       │
│  ❌ feast materialize              → Needs server pod       │
└─────────────────────────────────────────────────────────────┘
""")

### Data Scientist vs Admin Responsibilities

In [ ]:
print("""
┌───────────────────────────────┬─────────────────┬─────────────────────────────────┐
│ Task                          │ Who             │ How                             │
├───────────────────────────────┼─────────────────┼─────────────────────────────────┤
│ Deploy infrastructure         │ Admin           │ oc apply -k manifests/          │
│ Create feature definitions    │ Admin           │ features.py + feast apply       │
│ Materialize features          │ Admin           │ feast materialize               │
│ Retrieve online features      │ Data Scientist  │ store.get_online_features()     │
│ Retrieve historical features  │ Data Scientist  │ store.get_historical_features() │
│ Train ML models               │ Data Scientist  │ 02-training.ipynb               │
└───────────────────────────────┴─────────────────┴─────────────────────────────────┘
""")

In [ ]:
# Next Steps
print("""
### Next Steps

1. 02-training.ipynb → Train model using historical features
2. 03-inference.ipynb → Deploy model with KServe using online features

### Alternative: Direct Connection Mode

If you need to run 'feast apply' or 'feast materialize' from a notebook,
use 01-feast-features.ipynb which generates a direct-connection config.
""")

In [ ]:
# End of notebook